In [ ]:
# CELL 1 — Google Drive, packages, GPU check
from google.colab import drive
drive.mount('/content/drive')
!pip install -q timm==1.0.11 kagglehub
import torch, os
print('PyTorch', torch.__version__, '| CUDA', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > L4 GPU'
print('GPU:', torch.cuda.get_device_name(0), '| vCPUs:', os.cpu_count())

In [ ]:
# CELL 2 — Configuration (original Hybrid-K Phase 1 settings, unchanged) and imports
import os, re, json, time, random, zlib, base64, hashlib, platform
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
import torch, torch.nn as nn, timm, torchvision, sklearn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.amp import autocast, GradScaler
from torchvision import transforms
from scipy.optimize import minimize_scalar
from sklearn.metrics import roc_auc_score
from PIL import Image

MODEL = 'swin_tiny_patch4_window7_224'          # Model 1 only
PRETTY = 'Swin-Tiny'
OUT = Path('/content/drive/MyDrive/HybridK_Phase1_corrected')   # everything is saved here
for d in ['splits', 'fold_results', 'predictions', 'logs', 'checkpoints', 'summary', '_smoke_test']:
    (OUT / d).mkdir(parents=True, exist_ok=True)

CONFIG = {
    # unchanged from the original phase_1.py
    'image_size': 224, 'batch_size': 32, 'epochs': 15,
    'learning_rate': 1e-4, 'weight_decay': 1e-4, 'label_smoothing': 0.1, 'warmup_epochs': 2,
    'imagenet_mean': [0.485, 0.456, 0.406], 'imagenet_std': [0.229, 0.224, 0.225],
    'replication_seeds': [42, 123, 456], 'n_folds': 5, 'partition_seed': 42,
    'use_amp': False, 'pretrained': True, 'timm': timm.__version__,
    # corrected Phase 1 design
    'split': 'parent-image-grouped StratifiedGroupKFold(5, seed 42); calib = 1/8 of outer-train '
             'parent images (StratifiedGroupKFold(8, seed 42+k), first fold); fixed manifest',
    'final_model': 'last epoch (as original); no checkpoint selection',
    'epoch_monitoring_set': 'calib subset (logging only); test fold evaluated once after training',
    'temperature_fit': 'NLL on calib-subset logits, bounded [0.25, 5.0], always applied',
    'ece_bins': 15,
    # efficiency only (no effect on the protocol): images resized once and cached; more loader workers
    'resize_cached_once': True,
    'num_workers': min(4, os.cpu_count() or 2), 'pin_memory': True,
}
DEVICE = torch.device('cuda')
print(json.dumps({k: CONFIG[k] for k in ['epochs', 'batch_size', 'learning_rate', 'num_workers', 'timm']}, indent=1))

In [ ]:
# CELL 3 — Load SIPaKMeD (same Kaggle dataset as the original phase_1.py)
import kagglehub
DATA_ROOT = Path(kagglehub.dataset_download('prahladmehandiratta/cervical-cancer-largest-dataset-sipakmed'))
# If kagglehub fails: upload kaggle.json, download with the Kaggle CLI as in phase_1.py,
# then set DATA_ROOT = Path('/content/sipakmed') and re-run this cell.
IMAGE_EXT = {'.bmp', '.png', '.jpg', '.jpeg', '.tif', '.tiff'}
CROP_RE = re.compile(r'^(?P<parent>\d+)_(?P<cell>\d+)$')

def find_cells(root):
    """Cropped cells NNN_MM.* under any im_* class folder (raw Kaggle or flattened layout)."""
    rows = []
    for p in Path(root).rglob('*'):
        if p.is_file() and p.suffix.lower() in IMAGE_EXT and CROP_RE.match(p.stem):
            cls = next((q for q in reversed(p.relative_to(root).parts[:-1]) if q.startswith('im_')), None)
            if cls:
                rows.append({'class': cls, 'filename': p.name, 'path': str(p)})
    df = pd.DataFrame(rows)
    assert not df.duplicated(['class', 'filename']).any(), 'duplicate files'
    return df.sort_values(['class', 'filename'], kind='mergesort').reset_index(drop=True)

cells = find_cells(DATA_ROOT)
print(DATA_ROOT, '->', len(cells), 'cropped cells', dict(Counter(cells['class'])))
assert len(cells) == 4049, 'Expected 4,049 cropped cells'

In [ ]:
# CELL 4 — Corrected parent-image-grouped split (fixed manifest) + zero-overlap audit
# Role of every parent image in each outer fold k (T=train, C=calibration, X=test), exactly as in
# split_manifest.csv from the corrected Phase 1 code. Expanded to cells and checked by SHA-256.
SPLIT_B64 = (
    "eNqNnE2P5LgNhu/5LwFM6ruvzmURLLBA+tC3oDPpAI3MZhYzk8MC+fEp2bVjysXHrTrqlWSKIqmXLNnvv/79L79/+/fb19fvX76/"
    "f/rfssjT8+338qf3B0SfnlcfCU/PL8/Pq4PEDXl2kHR7zouLZJSg4Gy1I6uHtNts64uDyELrEUFESQIJTysg8WmF2VAHkvE5BcdU"
    "RNpTF8BBdCGpVRBReo6GvnOrs1KNTy+ruz+anvrGObajGZ/TdeCvp5LetNFsYaExQXCMdqlXR+oQSLYQ+y54dhA2O/C0EzJ5VkA7"
    "CBUlaORZcaExUZ66cjxEyedioPXE2D3rxVlPTE/dQDzZMB7EQnYdd1/wnoM6SKiDJIgoaScFfE5EZPcFb7bcV+p5Vio4G8aD1Ejq"
    "vCCCvpAxHuSASNeBu9sZdZAz7UJGHeRK0T+jDspCUpebDl5c2QraQQm0nhLJS0qi9ZRMPlfwbCwYE0sjL6kYEyvGg4p2UDEm1ohI"
    "QgRjYsWYWNEOKp4LbaHo0lAHDflBu+lg9Z8TcbZbTPTtrWWcDWNi2+KBd2Y18gVZ9pi4OojASmWhc0HuPNFDdh14zyFfkIX4gXSe"
    "uHo6kKXibGQHIhQTRVAHN564AkLngnSe6NqoSKJdkIyzFVxPRaTRc5TigSjqYOOJrmwaSKMaEenngmtVN55IY7oduBrtPNH1H+k8"
    "0V9PQDu48cTuph6CvhBQB50nuvFNAuULEtAOkCdK54m+l+w80UMi6iAKnGbSeeLqIwHyLNl54uohCRikRLSDiDqIux14s6EObjyx"
    "77YzJhE/EOSJcuOJq78LCWPixhP92TJkYHLjiT4PkRtPJKRBxiKZuLJkOhslbzHRs4NMubNsPNHduYy+kPFcQJ4ouaLUaAcbT3Rn"
    "6zzRZbdSlCzkzhOd3S4RJUA7KFQ/kFJwTH1aXTYoBWNiRTuoQtGlog4qno01IoLxoGaUrSCCdrDxRNezGtoB8kS58USfpUnnib7t"
    "tEincyOuLA3toBU6T1vF2Ygn6kL8QLGeqIvimADnqS4RZ0tgVbqQHehSECGOpEuDqKyywFmvgjoQypkUeaJuPNGVWigeKPJEFcob"
    "VSqOoZioSrmzKuoAeaKqqR/89cv75y+ffj+q1D9WekJ0i2IeEo4c8IREnM2wzhOSj9z5hOwZmIdUHGMi7IjsVuWNEVOVOiGKs5kK"
    "7QlBHUg6TucTknFMQakr7YK1qhGx2ccJMYzrhCjJdrMqsBCN+JyEs+WDw56QgmMqPucWXVZ3PTb7OCFCswXjWScEPSvE49w+IQmf"
    "k8mqQkEJKu12zz58jcblOBdOiBwx8YQoWUg0FZkTgr6wVan9MRllK0def0IqjmnkP2khjSa0g6Sk62SqkyckHpnRCUkoQSbt2Ozj"
    "hNQj2z0hjWw0m5P2hAiO0SM/PSGB1pPRDjLqIOO5kNEXciXtZIyJZUEEdbBVqd3nbNmHq9EScbZE/lMyrbQUlPpmBzBbo/Nnzz68"
    "uFPlyFxPyF6RcfbHVqlPSD8XVk87Fe3AZh8nxFSlTgjGxIp20NAOmpBsDflBzz78aNkizoY6aFs88PRms48TUimGtAaeJQvxg6FK"
    "fUIUbGeoUp8Qige9Sr3CGLKDvUr94o6p4HOyNJJAyA56lZrGED+4V6kdDis7T1y92RJKkGk9yBN7lRo0Ko2k1gW8XlRwDPFEUYoH"
    "vUpNsyWcDe1AC5xmW5Xa15uiL4SFxgRTlTohqAPkiVuV2rfekIBB3qvULlJQNjob9yq1K0HEeBApZ+pVap+LbVVqf3+QJ25VapCN"
    "8oVepSYJKq4H40Ha8gXnnJOEvrBXqb2VJuLKvUrtnxiCPPFepXYRtIOEvrDdZlg9X8joCxntICsi6AvIE3uVeoXZMuk6F5wNdYA8"
    "cahSnxDiiVuVGhD0hRIpkhfKG4cq9QkpFMlLxfXguVAX8rmKdlBRBzXgc9AOKtWR9iq1G8lrwTFoB7XRuY08UZAn9io1jQnH/3Mn"
    "JOIY5AcN40ErtNKG/KBRLa1Xqf0Iq8gTtyq1G/kU64mKPHGvUvtIhnNO7a3XE1IRaXAGq1A8UEEdIE/cqtQwJuKYhAjxAxWyg16l"
    "Bl2LqdSPyFaldndOKSbeq9SOxSvyRMV6omrCMRk8WLdbr77UFVfathtEDhLQDgLqICgiATI93W+9eusJqINAMVGxnqjB+MLPb99f"
    "f/v8+u3HhfY/CMoImOg2AuY6+whEekY6BB6BTCPMIT8CJq6NQIMRN49e/7hfMQImsI8Ardz+6TQC5r/HEUg0IhNgzHgETOY/Au0I"
    "wQNg/xcYAVq5Ko0w5G4EIuhK00EqRoBWroXWQbarDR4eFtCV/TNgBGjP7YX1EYiwDntdfQRo5aEc15dHoIJRB9rzaHL8ETB/AYxA"
    "v3eyOuLGAJZo87oRSARk8NpI1h4rGFxssLX2fvoICEiVzJk1AuTnKcI6EkW4lOnhhZ5RCWhHmXUAbB43AuaSyQgoPCObBGYEzI30"
    "EUjHX3IjQCvPFNsz7XluoN2eva2eVIX8vJgcfgQCSGUr/CNA1r7lbZ5KCq280Mpt0jYAlVa+X0B3FlgVIkOlCFcjBJmaQLs1wzr2"
    "K0WOq9ma/gg0mKqRtTcBK+l5WqcAjwDteaMI19Lxt98I0MobrbxRbN+vET3u4P22+eNUsoivRFngPO9FfHhGpKnSQaJHINMzwNr3"
    "+r1ju2LTsgEQiO1CHE7udzwexbUp2QhEmgrOcxGIcGIvDY1AJakajFCw9r1k7ylR9bh/OwKBRkSfqg2Xykcgw34orVwrjWhHajQA"
    "YfE5tRCH22+Trx5A1h7Az/er5J5KbhxudU20czhfqkri0p7bW+QjIAQoWEkMBEDGspfmvZVH2vNYaKpKU8Gpdq/KO+IminCJVp4g"
    "S+0V+dW19gSn2laP94FCUlWSCrLU4cb4CAhoNyuNCPSM6FPn4bL4CFCEy4VGQH7eS/A/iosDUGjlxOHEvk04Aua6zgjQqVYSAdln"
    "S7307p9Rhax943Be3N3uZ3ihr9LKK1l7hYxluxnuT5XArmqG06BSbCcOJ8Thtmq7O1UTGqE+F90vhHviNtrzBry919kBgCx1uwvu"
    "6qpBbN9vgq8eAExmq7B7KtkL7M4ClepwusDKdcn08OKzvl5bB6D5tYzh/vcIgLX3urovrgTfcZQ43FZU95+RfbvqJXW3XjJU1EfA"
    "/NE6ANu9i/ubpSMA1j7c+R4BiO1bLd0zUVXac83+adAL6as/Fa18v27hLDCAn/ciuj9VoD0PtHL73YgRSH66pAGYzF4+X70R1U8t"
    "NUCE037HwrWrSH4eaeXE4TRChFPicEocTmOhh1caARxOE+15kuMq6wjQylOgqSKY6FaHcwHIzzUVegatPFFsz7TnWXwuqlkJoD3P"
    "FNs7h1u9ledMI4C9audwq+c4Gertam9RjICpRv3y+vX1H6/fXj+PX1GyzeZ1YNtslGGbzYs+tjn5vc3hbpuL32wSddtstt40y+I+"
    "0t4stM3mQ1G2ORylJdsc/UmSL4lxcNvsr9IeZba5uc26uHOrv0o1rzXb5uBusb0yaJuTP3d29a3Fb66uqtTQctMcTGHVNsvBd2yz"
    "unIH8w6NbTYFVduc/Emy31zcbbAvjNjm5jbvJYaHbYjiqiqq3+z75X4srWe5o/lyjW32/TIW/5H+XtrDyDQn82eYbRY3cCR1Ny0F"
    "V8AUXUluB9DqeHHy9zL5q0yGbthmf5XbR4keFZvNVxdss7qblv1VZlMmss3J9eKc3Z3PvsVm32KzH32KH32K+M3qyl38VRb/JCn+"
    "SWKv5dlmP/psr407c/t+aT8sZJt9v6y+X9bgxqoaXQGrv5d7GeBRQN9ia3VNs7Yj2zXNzT8vm7gRufkW2/+8We/fmLPN/klib9fZ"
    "Zv+8bP4qW/UlaV6Y2f+ueejd/6xZH53k/sLF4yTBE1AWN/ps71msTu/sKXZ7w8JrNq+g/u2/v719/df7p/fXz3/+6T/f377++vbP"
    "99fvb3cKd8+VLnqZVV30ClO94lQvc8xc9MpTcxmXvuhVj7P0opdxeO4l5uLKRS85XPail85Ib/8Pu+gVp3qZWxAXvUzJ5aJXmdJX"
    "nVpjm9ltNX8+XPQyHz+96KUz+2hvVV30Mh/Cu+iVptaYZ7SqZWou80GUi15txlY31v2hXEFmNHG/hvmBXGHK7sOU3d9f/f7oieZa"
    "yEWvMvXEOuNDwaQ43Mt+3PSil8zIFc0lk4teU3Yfp3Qfp+w+TsX7WKakrzP2Fad0n8yVjoteMiN9mor3aeqsTXFqrjQT5dJUzEnl"
    "+PzVRa86E6P3f1nXD3YoLzM8J8tRarzopTO2msPxMaaLXubDZhe90owmtozsY+mneE6uMyefTd+4V5k6a2/J3fpxlCtTdm9fzbro"
    "FafkmtJ9yTP6KlM8p0xx8jLFc+oy1Utmnlh1RhM1HHdEL3rFGX3VKd3XPNWrTPWqM75tXy3jXm2ZYQpNZmyiTdl9C1NzTcWclmY0"
    "Ya9CXvQyr/Ne9JqK920mt+r5+MfRRJaZs7Zn6x/LNSTvF73ixCk6pPYXvfKEfckywzGHzyxc9JrSvczE++E250UvnYjk/Z7nzBPj"
    "lPRpwjtE8pRcZUqumZgz3BrlXjrD74cPFV/00olTQTRMzTWle/sq4EWve7z/P1yJn0k="
)
MANIFEST_SHA256 = '01163b7112fcd935bac3941b8b64489125e7e417c39765d650c2711adcdb3864'
ROLE = {'T': 'train', 'C': 'calib', 'X': 'test'}

group_roles = dict(l.split(':') for l in zlib.decompress(base64.b64decode(SPLIT_B64)).decode().splitlines())
man = cells.copy()
man['parent_id'] = man['filename'].str.extract(r'^(\d+)_\d+\.')[0]
man['group_id'] = man['class'] + '|' + man['parent_id']
classes = sorted(man['class'].unique())
man['label'] = man['class'].map({c: i for i, c in enumerate(classes)})
assert set(man['group_id']) == set(group_roles), 'Parent images on disk differ from the manifest'
for k in range(5):
    man[f'role_fold{k}'] = man['group_id'].map(lambda g: ROLE[group_roles[g][k]])
man['test_fold'] = man['group_id'].map(lambda g: group_roles[g].index('X'))
canon = man[['class', 'filename', 'parent_id', 'group_id', 'test_fold'] + [f'role_fold{k}' for k in range(5)]] \
    .sort_values(['class', 'filename']).to_csv(index=False)
assert hashlib.sha256(canon.encode()).hexdigest() == MANIFEST_SHA256, 'Manifest hash mismatch: do not train'

rows = []
for k in range(5):
    r = man[f'role_fold{k}']
    G = {s: set(man.loc[r == s, 'group_id']) for s in ('train', 'calib', 'test')}
    ov = (len(G['train'] & G['test']), len(G['calib'] & G['test']), len(G['train'] & G['calib']))
    assert ov == (0, 0, 0), f'fold {k}: parent overlap {ov}'
    rows.append({'fold': k, **{f'{s}_cells': int((r == s).sum()) for s in ('train', 'calib', 'test')},
                 **{f'{s}_parents': len(G[s]) for s in ('train', 'calib', 'test')},
                 'overlap_train_test': ov[0], 'overlap_calib_test': ov[1], 'overlap_train_calib': ov[2]})
audit = pd.DataFrame(rows)
print(audit.to_string(index=False))
print(f'\nManifest verified (SHA-256 match). {len(man)} cells, {man.group_id.nunique()} parent images, '
      'ZERO parent-image overlap in all folds. (Parent-image IDs are not patient IDs.)')
man.drop(columns=['path']).to_csv(OUT / 'splits' / 'split_manifest_verified.csv', index=False)
audit.to_csv(OUT / 'splits' / 'split_audit_per_fold.csv', index=False)
json.dump({'config': CONFIG, 'model': MODEL, 'manifest_sha256': MANIFEST_SHA256,
           'env': {'python': platform.python_version(), 'torch': torch.__version__,
                   'torchvision': torchvision.__version__, 'timm': timm.__version__,
                   'sklearn': sklearn.__version__, 'gpu': torch.cuda.get_device_name(0)}},
          open(OUT / f'run_config__{MODEL}.json', 'w'), indent=2)

In [ ]:
# CELL 5 — Metrics, calibration, data pipeline, one training run (all original settings)
# ---- eight Hybrid-K metrics: copied from sipakmed_full_pipeline_all_in_one.py ----
def _clean_probs(p):
    p = np.nan_to_num(np.asarray(p, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    p = np.clip(p, 0.0, None); s = p.sum(1, keepdims=True); p = p / np.where(s < 1e-12, 1.0, s)
    z = p.sum(1) < 1e-12
    if z.any(): p[z] = 1.0 / p.shape[1]
    return p
def m_acc(y, p): return float((_clean_probs(p).argmax(1) == y).mean())
def m_f1(y, p):
    p = _clean_probs(p); pr_ = p.argmax(1); f = []
    for c in range(p.shape[1]):
        tp = ((pr_ == c) & (y == c)).sum(); fp = ((pr_ == c) & (y != c)).sum(); fn = ((pr_ != c) & (y == c)).sum()
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        f.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return float(np.mean(f))
def _ece_on(conf, corr, nb=15):
    b = np.linspace(0, 1, nb + 1); e = 0.0; N = len(conf)
    for i in range(nb):
        mm = (conf > b[i]) & (conf <= b[i + 1])
        if mm.sum(): e += (mm.sum() / N) * abs(corr[mm].mean() - conf[mm].mean())
    return e
def m_ece(y, p):
    p = _clean_probs(p); return float(_ece_on(p.max(1), (p.argmax(1) == y).astype(float)))
def m_wc_ece(y, p):
    p = _clean_probs(p); v = []
    for c in range(p.shape[1]):
        m = y == c
        v.append(_ece_on(p[m].max(1), (p[m].argmax(1) == y[m]).astype(float)) if m.sum() else np.nan)
    return float(np.nanmax(v))
def m_brier(y, p):
    p = _clean_probs(p); return float(np.mean(np.sum((p - np.eye(p.shape[1])[y]) ** 2, 1)))
def m_nll(y, p):
    p = _clean_probs(p); return float(-np.mean(np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1.0))))
def m_aurc(y, p):
    p = _clean_probs(p); conf = p.max(1); corr = (p.argmax(1) == y).astype(float); o = np.argsort(-conf)
    risk = 1 - np.cumsum(corr[o]) / np.arange(1, len(y) + 1); cov = np.arange(1, len(y) + 1) / len(y)
    return float((np.trapezoid if hasattr(np, 'trapezoid') else np.trapz)(risk, cov))
def m_auroc(y, p):
    try: return float(roc_auc_score(y, _clean_probs(p), multi_class='ovr', average='macro'))
    except Exception: return float('nan')
METRICS = {'acc': m_acc, 'f1': m_f1, 'auroc': m_auroc, 'ece': m_ece, 'wc_ece': m_wc_ece,
           'aurc': m_aurc, 'brier': m_brier, 'nll': m_nll}
def all_metrics(y, p): return {k: f(y, p) for k, f in METRICS.items()}

# ---- temperature scaling: same objective/bounds as the original fitT, fitted on calib subset only ----
def softmax_T(lg, T=1.0):
    z = lg / T; z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
def fit_temperature(lg, y):
    def nll(T):
        z = lg / T; z = z - z.max(1, keepdims=True)
        return -(z[np.arange(len(y)), y] - np.log(np.exp(z).sum(1))).mean()
    return float(minimize_scalar(nll, bounds=(0.25, 5.0), method='bounded').x)

# ---- reproducibility (as original) ----
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
def seed_worker(_):
    s = torch.initial_seed() % 2 ** 32; np.random.seed(s); random.seed(s)

# ---- transforms (original). Resize((224,224)) is deterministic and was the FIRST step of both
#      original pipelines, so it is applied once when caching; the remaining steps are identical. ----
norm = transforms.Normalize(mean=CONFIG['imagenet_mean'], std=CONFIG['imagenet_std'])
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5), transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ToTensor(), norm])
eval_tf = transforms.Compose([transforms.ToTensor(), norm])

class View(Dataset):
    def __init__(self, imgs, labels, idx, tf): self.imgs, self.labels, self.idx, self.tf = imgs, labels, idx, tf
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]; return self.tf(self.imgs[j]), int(self.labels[j])

@torch.no_grad()
def predict_logits(model, loader):
    model.eval(); L, Y = [], []
    for x, y in loader:
        with autocast('cuda', enabled=CONFIG['use_amp']):
            L.append(model(x.to(DEVICE, non_blocking=True)).float().cpu().numpy())
        Y.append(y.numpy())
    return np.concatenate(L), np.concatenate(Y)

def tag_of(seed, fold): return f'multiclass__{MODEL}__rep{seed}__fold{fold}'

def train_one_unit(seed, fold, out_dir, epochs, limit=None):
    """One fold x seed. Saves everything to Drive; the JSON is written LAST (atomically),
    so a JSON exists only for a fully completed run."""
    labels = man['label'].to_numpy(); role = man[f'role_fold{fold}'].to_numpy()
    tr, ca, te = [np.where(role == r)[0] for r in ('train', 'calib', 'test')]
    if limit:   # smoke test only
        rng = np.random.default_rng(0)
        tr, ca, te = [np.sort(rng.choice(a, min(limit, len(a)), replace=False)) for a in (tr, ca, te)]
    set_seed(seed)
    w = torch.tensor(1.0 / np.bincount(labels[tr], minlength=5)[labels[tr]], dtype=torch.double)
    g = torch.Generator(); g.manual_seed(seed)
    sampler = torch.utils.data.WeightedRandomSampler(w, num_samples=len(tr), replacement=True, generator=g)
    kw = dict(num_workers=CONFIG['num_workers'], pin_memory=True, persistent_workers=True)
    tr_ld = DataLoader(View(IMAGES, labels, tr, train_tf), batch_size=CONFIG['batch_size'], sampler=sampler,
                       worker_init_fn=seed_worker, generator=g, **kw)
    ca_ld = DataLoader(View(IMAGES, labels, ca, eval_tf), batch_size=CONFIG['batch_size'], shuffle=False, **kw)
    te_ld = DataLoader(View(IMAGES, labels, te, eval_tf), batch_size=CONFIG['batch_size'], shuffle=False, **kw)

    model = timm.create_model(MODEL, pretrained=True, num_classes=5).to(DEVICE)
    opt = AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
    spe = len(tr_ld); wu = max(1, CONFIG['warmup_epochs'] * spe); dec = max(1, (epochs - CONFIG['warmup_epochs']) * spe)
    sched = SequentialLR(opt, [LinearLR(opt, start_factor=0.01, end_factor=1.0, total_iters=wu),
                               CosineAnnealingLR(opt, T_max=dec)], milestones=[wu])
    crit = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])
    scaler = GradScaler('cuda', enabled=CONFIG['use_amp'])
    hist, t0 = [], time.time()
    for ep in range(epochs):
        model.train(); sl = sc = sn = 0
        for x, y in tr_ld:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad()
            with autocast('cuda', enabled=CONFIG['use_amp']):
                lg = model(x); loss = crit(lg, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            sl += loss.item() * x.size(0); sc += (lg.argmax(1) == y).sum().item(); sn += y.size(0)
        cl, cy = predict_logits(model, ca_ld)
        rec = {'epoch': ep + 1, 'train_loss': sl / sn, 'train_acc': sc / sn,
               'calib_loss': float(crit(torch.tensor(cl), torch.tensor(cy))), 'calib_acc': float((cl.argmax(1) == cy).mean()),
               'lr_end': opt.param_groups[0]['lr'], 'elapsed_s': round(time.time() - t0, 1)}
        hist.append(rec)
        print(f"   ep {ep+1:2d}/{epochs} tr_loss={rec['train_loss']:.4f} tr_acc={rec['train_acc']:.4f} "
              f"calib_loss={rec['calib_loss']:.4f} calib_acc={rec['calib_acc']:.4f} [{rec['elapsed_s']:.0f}s]")
        assert np.isfinite(rec['train_loss']), 'non-finite loss'
    train_time = time.time() - t0

    # final model = last epoch; test fold used ONCE, here
    cal_lg, cal_y = predict_logits(model, ca_ld); te_lg, te_y = predict_logits(model, te_ld)
    assert np.isfinite(cal_lg).all() and np.isfinite(te_lg).all(), 'non-finite logits'
    T = fit_temperature(cal_lg, cal_y)                          # calibration subset only
    p_raw, p_cal = softmax_T(te_lg), softmax_T(te_lg, T)
    met_raw, met_cal = all_metrics(te_y, p_raw), all_metrics(te_y, p_cal)

    tag = tag_of(seed, fold); out_dir = Path(out_dir)
    for d in ['fold_results', 'predictions', 'logs', 'checkpoints']:
        (out_dir / d).mkdir(parents=True, exist_ok=True)
    R, C = man.iloc[te], man.iloc[ca]
    pd.DataFrame(hist).to_csv(out_dir / 'logs' / f'{tag}.csv', index=False)
    pred = pd.DataFrame({'filename': R.filename.values, 'class': R['class'].values, 'parent_id': R.parent_id.values,
                         'group_id': R.group_id.values, 'y_true': te_y})
    for c in range(5): pred[f'logit_{c}'] = te_lg[:, c]
    for c in range(5): pred[f'p_raw_{c}'] = p_raw[:, c]
    for c in range(5): pred[f'p_cal_{c}'] = p_cal[:, c]
    pred.to_csv(out_dir / 'predictions' / f'{tag}__test.csv', index=False)
    cpred = pd.DataFrame({'filename': C.filename.values, 'class': C['class'].values, 'group_id': C.group_id.values, 'y_true': cal_y})
    for c in range(5): cpred[f'logit_{c}'] = cal_lg[:, c]
    cpred.to_csv(out_dir / 'predictions' / f'{tag}__calib.csv', index=False)
    ck_tmp = out_dir / 'checkpoints' / f'{tag}.pt.tmp'
    torch.save({'state_dict': model.state_dict(), 'model_name': MODEL, 'seed': seed, 'fold': fold,
                'temperature': T, 'config': CONFIG}, ck_tmp)
    os.replace(ck_tmp, out_dir / 'checkpoints' / f'{tag}.pt')
    record = {
        # same keys/semantics as the original fold JSONs (y_prob = raw softmax)
        'y_true': te_y.astype(int).tolist(), 'y_pred': p_raw.argmax(1).astype(int).tolist(),
        'y_prob': p_raw.tolist(), 'model_name': MODEL, 'task_mode': 'multiclass',
        'replication_seed': seed, 'fold_idx': fold, 'cv_seed': CONFIG['partition_seed'],
        'accuracy': met_raw['acc'], 'f1_score': met_raw['f1'], 'roc_auc': met_raw['auroc'],
        # added for leakage-free Phase 2
        'logits': te_lg.tolist(), 'temperature': T, 'temperature_fit_on': 'calib subset of this model/fold/seed',
        'y_prob_calibrated': p_cal.tolist(),
        'filenames': R.filename.tolist(), 'classes': R['class'].tolist(),
        'parent_ids': R.parent_id.tolist(), 'group_ids': R.group_id.tolist(),
        'calib': {'filenames': C.filename.tolist(), 'classes': C['class'].tolist(), 'group_ids': C.group_id.tolist(),
                  'y_true': cal_y.astype(int).tolist(), 'logits': cal_lg.tolist()},
        'metrics_test_raw': met_raw, 'metrics_test_calibrated': met_cal,
        'metrics_calib_subset': {'raw': all_metrics(cal_y, softmax_T(cal_lg)), 'calibrated': all_metrics(cal_y, softmax_T(cal_lg, T))},
        'split': {'n_train': len(tr), 'n_calib': len(ca), 'n_test': len(te),
                  'n_train_parents': int(man.iloc[tr].group_id.nunique()), 'n_calib_parents': int(C.group_id.nunique()),
                  'n_test_parents': int(R.group_id.nunique()), 'manifest_sha256': MANIFEST_SHA256,
                  'manifest_column': f'role_fold{fold}', 'partition_seed': CONFIG['partition_seed'], 'training_seed': seed},
        'history': hist, 'train_time_seconds': train_time, 'n_params': int(sum(p.numel() for p in model.parameters())),
        'config': CONFIG, 'subset_limit': limit, 'class_names': classes,
    }
    js_tmp = out_dir / 'fold_results' / f'{tag}.json.tmp'
    json.dump(record, open(js_tmp, 'w'))
    os.replace(js_tmp, out_dir / 'fold_results' / f'{tag}.json')   # written last
    del model, opt; torch.cuda.empty_cache()
    return record

# ---- verification of all 15 runs and files ----
REQUIRED = ['y_true', 'y_pred', 'y_prob', 'logits', 'temperature', 'y_prob_calibrated', 'filenames', 'parent_ids',
            'group_ids', 'calib', 'metrics_test_raw', 'metrics_test_calibrated', 'split', 'history', 'config']
def verify_unit(seed, fold, out_dir=OUT):
    tag = tag_of(seed, fold); o = Path(out_dir); probs = []
    files = {'json': o / 'fold_results' / f'{tag}.json', 'test_csv': o / 'predictions' / f'{tag}__test.csv',
             'calib_csv': o / 'predictions' / f'{tag}__calib.csv', 'log_csv': o / 'logs' / f'{tag}.csv',
             'checkpoint': o / 'checkpoints' / f'{tag}.pt'}
    for n, f in files.items():
        if not f.exists() or f.stat().st_size == 0: probs.append(f'missing {n}')
    if files['json'].exists():
        try:
            d = json.load(open(files['json']))
            miss = [k for k in REQUIRED if k not in d]
            if miss: probs.append(f'missing keys {miss}')
            te = man[man[f'role_fold{fold}'] == 'test']; ca = man[man[f'role_fold{fold}'] == 'calib']
            if d.get('subset_limit'): probs.append('smoke-test subset')
            if sorted(d['filenames']) != sorted(te.filename) or sorted(d['group_ids']) != sorted(te.group_id):
                probs.append('test cells differ from manifest')
            if sorted(d['calib']['group_ids']) != sorted(ca.group_id): probs.append('calib cells differ from manifest')
            n = len(d['y_true'])
            if not (len(d['y_prob']) == len(d['logits']) == len(d['y_prob_calibrated']) == n): probs.append('length mismatch')
            if len(d['history']) != CONFIG['epochs']: probs.append(f"{len(d['history'])} epochs logged")
            v = list(d['metrics_test_raw'].values()) + list(d['metrics_test_calibrated'].values())
            if len(v) != 16 or not all(np.isfinite(v)) or not np.isfinite(d['temperature']):
                probs.append('missing/non-finite metrics or temperature')
        except Exception as e:
            probs.append(f'unreadable json: {e}')
    return probs

In [ ]:
# CELL 6 — Cache images in RAM, resized once to 224x224 (identical to the original first transform step)
t = time.time(); IMAGES = []
for p in man['path']:
    with Image.open(p) as im:
        IMAGES.append(im.convert('RGB').resize((CONFIG['image_size'], CONFIG['image_size']), Image.BILINEAR))
print(f'Cached {len(IMAGES)} images in {time.time() - t:.0f}s')
# (torchvision Resize on PIL images uses bilinear interpolation, i.e. the same operation)

In [ ]:
# CELL 7 — SMOKE TEST (≈2–4 min): 1 epoch, fold 0, seed 42, 64 cells per role -> _smoke_test/ (not results)
r = train_one_unit(seed=42, fold=0, out_dir=OUT / '_smoke_test', epochs=1, limit=64)
print('SMOKE TEST OK | T =', round(r['temperature'], 3), '| files in', OUT / '_smoke_test')
print('Use the epoch time printed above x 15 epochs x 15 runs (scaled by ~44x the smoke-test cells) only as a rough guide;')
print('the first full run below gives the reliable time per run.')

In [ ]:
# CELL 8 — FULL TRAINING: Swin-Tiny, 5 folds x 3 seeds = 15 runs (resumable: completed runs are skipped)
units = [(s, k) for s in CONFIG['replication_seeds'] for k in range(CONFIG['n_folds'])]
t_all = time.time()
for i, (s, k) in enumerate(units, 1):
    if not verify_unit(s, k):
        print(f'[skip] {i}/15 seed {s} fold {k}: already complete and verified'); continue
    print(f'\n>> {PRETTY} | seed {s} | fold {k} | run {i}/15')
    t = time.time()
    r = train_one_unit(s, k, OUT, epochs=CONFIG['epochs'])
    left = verify_unit(s, k)
    assert not left, f'Saved files incomplete for seed {s} fold {k}: {left}'
    print(f"   saved + verified in {(time.time() - t) / 60:.1f} min | T={r['temperature']:.3f} | "
          f"test acc={r['metrics_test_raw']['acc']:.4f} | ECE raw/cal="
          f"{r['metrics_test_raw']['ece']:.4f}/{r['metrics_test_calibrated']['ece']:.4f}")
print(f'\nSession training time: {(time.time() - t_all) / 60:.1f} min')

In [ ]:
# CELL 9 — FINAL VERIFICATION of all 15 runs + per-run metric summary (saved to Drive)
ok, rows = 0, []
for s in CONFIG['replication_seeds']:
    for k in range(CONFIG['n_folds']):
        p = verify_unit(s, k)
        print(f'seed {s} fold {k}: ' + ('OK' if not p else 'INCOMPLETE: ' + '; '.join(p)))
        ok += not p
        if not p:
            d = json.load(open(OUT / 'fold_results' / f'{tag_of(s, k)}.json'))
            for kind in ('raw', 'calibrated'):
                rows.append({'model': PRETTY, 'seed': s, 'fold': k, 'kind': kind, 'temperature': d['temperature'],
                             **d[f'metrics_test_{kind}']})
print(f'\n{ok}/15 runs complete for {PRETTY}.')
if rows:
    df = pd.DataFrame(rows)
    df.to_csv(OUT / 'summary' / f'per_run_metrics__{MODEL}.csv', index=False)
    agg = df.groupby('kind')[list(METRICS) + ['temperature']].agg(['mean', 'std'])
    agg.to_csv(OUT / 'summary' / f'mean_sd__{MODEL}.csv')
    print(agg.round(4).T.to_string())
print('ALL 15 RUNS VERIFIED. Model 1 is complete.' if ok == 15 else
      'NOT COMPLETE: re-run Cells 1-6 and Cell 8; completed runs are skipped.')
print('Descriptive per-model results only: no ranking or ensemble selection is done in Phase 1.')